# RigTech · runner_colab_continuacao — DINOv3 + Transformer (per-crop binário)

**Filosofia do ciclo**: o modelo é **instrumento fixo de auditoria de rótulos** — mesmo backbone (DINOv3-ViTL16 satélite, congelado), mesma head (2 layers Transformer sobre patch tokens + mean pool), mesmos hiperparâmetros. O que muda entre ciclos é o **dataset** (correções humanas em `Datasets/DaninhasTreinoClientes/`). Cada rodada produz artefatos versionados (`_c1.pt`, `_c2.pt`, ...) pra medir quanto as correções melhoraram o rótulo.

**Task**: classificação **binária** por **crop 224×224**:
- `0` = cultivo
- `1` = daninha  (folha_larga + folha_estreita + mamona colapsadas)

**Amostragem**:
- **Positivos**: 1 crop centrado no centroide de cada polígono de daninha anotado (qualquer tipo).
- **Negativos**: até `MAX_CULTIVO_CROPS_PER_SITE` crops sampleados aleatoriamente da plantação (respeitando `INVERTED_PLANTACAO` pra Flaviano), sem interseção com daninha.

**Uso pra achar rótulos ruins** (só sobre os positivos anotados):
- `misclass`: modelo prediz `cultivo` num polígono anotado como daninha → provavelmente o polígono está errado (ou é sliver, ou área sem daninha real).
- `low_conf`: `prob_daninha < SUSPECT_LOW_CONF_THRESHOLD` — geometria ambígua ou marginal.

**Pré-requisitos no Drive**:
- `MyDrive/Datasets/DaninhasTreinoClientes/{Giasa,DoisRiosFlaviano,Flaviano01,CelsoSTE2,Celso01}/{imagem,daninhas,plantacao}`.
- Secret `HF_TOKEN` no Colab (chave lateral) — `dinov3-vitl16-pretrain-sat493m` é *gated*; aceitar termos em `huggingface.co/facebook/dinov3-vitl16-pretrain-sat493m`.

Runtime → GPU **A100** (T4 funciona, só demora mais).

## 1. Montar Drive e instalar dependências

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install rasterio geopandas scikit-image joblib tqdm scikit-learn shapely transformers torch huggingface_hub

## 2. Imports

In [ ]:
import os
import csv
import math
import hashlib
import inspect
import numpy as np
import rasterio
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import joblib
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

## 3. Login no Hugging Face

O checkpoint `dinov3-vitl16-pretrain-sat493m` é gated. Aceite os termos e coloque o token em *Secrets* do Colab como `HF_TOKEN`.

In [ ]:
from huggingface_hub import login

_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')

if _hf_token:
    login(token=_hf_token)
    print('Login no Hugging Face OK (via HF_TOKEN).')
else:
    login()

## 4. Configuração

**Filosofia do ciclo**: NADA aqui muda entre rodadas — só `CICLO` incrementa pra versionar os artefatos.

In [ ]:
# ---- ciclo (versiona artefatos, nao altera treino) ----
CICLO = 1

BASE = '/content/drive/MyDrive/Datasets/DaninhasTreinoClientes'

# Sites: (nome, imagem, [geojsons_de_daninha], plantacao). TODOS os geojsons de
# daninha viram classe 1, independente do tipo (folha_larga/folha_estreita/mamona).
PARES_CONFIG = [
    {
        'nome': 'Giasa',
        'imagem': f'{BASE}/Giasa/imagem/Giasa.tif',
        'geojsons': [
            f'{BASE}/Giasa/daninhas/FolhaLargaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/FolhaEstreitaGiasa (1).geojson',
            f'{BASE}/Giasa/daninhas/MamonasGiasa (1).geojson',
        ],
        'plantacao': f'{BASE}/Giasa/plantacao/Giasa_plantacao.geojson',
    },
    {
        'nome': 'DoisRiosFlaviano',
        'imagem': f'{BASE}/DoisRiosFlaviano/imagem/DoisRiosFlaviano.tif',
        'geojsons': [
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaLargaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/FolhaEstreitaDoisRiosFlaviano.geojson',
            f'{BASE}/DoisRiosFlaviano/daninhas/MamonasDoisRiosFlaviano.geojson',
        ],
        'plantacao': f'{BASE}/DoisRiosFlaviano/plantacao/DoisRiosFlaviano_plantacao.geojson',
    },
    {
        'nome': 'Flaviano',
        'imagem': f'{BASE}/Flaviano01/imagem/Flaviano01.tif',
        'geojsons': [
            f'{BASE}/Flaviano01/daninhas/FolhaLargaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/FolhaEstreitaFlaviano-1 (1).geojson',
            f'{BASE}/Flaviano01/daninhas/MamonasFlaviano-1 (1).geojson',
        ],
        'plantacao': f'{BASE}/Flaviano01/plantacao/Flaviano_plantacao.geojson',
    },
    {
        'nome': 'CelsoSTE2',
        'imagem': f'{BASE}/CelsoSTE2/imagem/CelsoSTE2.tif',
        'geojsons': [
            f'{BASE}/CelsoSTE2/daninhas/FolhaLargaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/FolhaEstreitaCelsoSTE-2 (1).geojson',
            f'{BASE}/CelsoSTE2/daninhas/MamonasCelsoSTE-2 (1).geojson',
        ],
        'plantacao': f'{BASE}/CelsoSTE2/plantacao/CelsoSTE2_plantacao.geojson',
    },
    {
        'nome': 'Celso01',
        'imagem': f'{BASE}/Celso01/imagem/Celso01.tif',
        'geojsons': [
            f'{BASE}/Celso01/daninhas/FolhasLargas_Celso_01 (1).geojson',
            f'{BASE}/Celso01/daninhas/FolhaEstreita_Celso_01 (1).geojson',
        ],
        'plantacao': f'{BASE}/Celso01/plantacao/Celso01_plantacao.geojson',
    },
]

# So o Flaviano tem o poligono de 'plantacao' com semantica invertida (marca a
# area SEM interesse). Confirmado na sessao do notebook base.
INVERTED_PLANTACAO = {'Flaviano'}

CLASS_NAMES = {0: 'cultivo', 1: 'daninha'}
N_CLASSES = 2

# ---- DINOv3 (variante satelite) ----
DINO_MODEL = 'facebook/dinov3-vitl16-pretrain-sat493m'
CROP_PIXELS = 224     # tamanho do crop enviado ao DINOv3 (multiplo de PATCH)
PATCH = 16            # grade 14x14 = 196 tokens por crop

# ---- Amostragem ----
MIN_POLYGON_PIXELS = 25            # descarta slivers minusculos
MAX_CULTIVO_CROPS_PER_SITE = 1500  # crops negativos (cultivo) por site

# ---- Split treino/validacao: faixa espacial dentro de CADA imagem ----
VAL_FRACTION = 0.20
MARGIN_TILES = 1       # gap de 1 x CROP_PIXELS entre treino e val (evita vazamento)
SPLIT_AXIS = 'x'

# ---- Transformer head ----
TRANSFORMER_LAYERS = 2
NHEAD = 8              # 1024 / 8 = 128 por cabeca
TRANSFORMER_DROPOUT = 0.15
CLASSIFIER_DROPOUT = 0.2

EPOCHS = 25
LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
BATCH = 64
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 7
RANDOM_STATE = 42

# ---- Loss: peso automatico pra classe minoritaria + label smoothing ----
CLASS_WEIGHTS = None       # None -> calculado por 1/freq no treino
LABEL_SMOOTHING = 0.05

# ---- Suspeitas ----
SUSPECT_LOW_CONF_THRESHOLD = 0.6   # positivo com prob_daninha < isso vira low_conf

# ---- Saidas versionadas por ciclo ----
OUTPUT_MODEL = f'/content/drive/MyDrive/modelo_dinov3_daninhabin_c{CICLO}.pt'
REPORT_PATH  = f'/content/drive/MyDrive/relatorio_dinov3_daninhabin_c{CICLO}.txt'
SUSPECTS_CSV = f'/content/drive/MyDrive/suspeitas_c{CICLO}.csv'
# Cache de features: NAO depende do ciclo. Assinatura por site inclui mtime dos
# geojsons -- se voce mudar um site, so ele recalcula.
CHECKPOINT_DIR = '/content/drive/MyDrive/dinov3_daninhabin_checkpoint'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Ciclo:', CICLO, '| output:', OUTPUT_MODEL)

## 5. Carregar DINOv3 (congelado)

Backbone congelado (`eval`, `no_grad`, na GPU). Normalização lida do `AutoImageProcessor` do checkpoint satélite. `interpolate_pos_encoding=True` pra tolerar variação do grid.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE, (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''))

proc = AutoImageProcessor.from_pretrained(DINO_MODEL)
DINOV3_MEAN = [float(x) for x in proc.image_mean]
DINOV3_STD  = [float(x) for x in proc.image_std]
print('mean:', DINOV3_MEAN, ' std:', DINOV3_STD)

print('Carregando DINOv3 (congelado):', DINO_MODEL)
model = AutoModel.from_pretrained(DINO_MODEL)
model.eval()
for p in model.parameters():
    p.requires_grad = False
model = model.to(DEVICE)

PATCH_SIZE = model.config.patch_size
NUM_REGISTER_TOKENS = getattr(model.config, 'num_register_tokens', 0)
HIDDEN_SIZE = model.config.hidden_size
print(f'patch_size={PATCH_SIZE} hidden_size={HIDDEN_SIZE} num_register_tokens={NUM_REGISTER_TOKENS}')
assert PATCH_SIZE == PATCH
assert CROP_PIXELS % PATCH == 0

mean_t = torch.tensor(DINOV3_MEAN, device=DEVICE).view(1, 3, 1, 1)
std_t  = torch.tensor(DINOV3_STD,  device=DEVICE).view(1, 3, 1, 1)
SUPPORTS_INTERP = 'interpolate_pos_encoding' in inspect.signature(model.forward).parameters
N_TOKENS = (CROP_PIXELS // PATCH) ** 2
print(f'crop {CROP_PIXELS}x{CROP_PIXELS} -> grid {CROP_PIXELS//PATCH}x{CROP_PIXELS//PATCH} = {N_TOKENS} tokens')

In [ ]:
@torch.no_grad()
def extract_crop_features_batch(crops_uint8_bhw3):
    """crops (B, H, W, 3) uint8 -> (B, N_TOKENS, HIDDEN_SIZE) float32 numpy.
    Descarta CLS + register tokens ANTES do reshape."""
    t = torch.from_numpy(crops_uint8_bhw3.astype(np.float32) / 255.0).permute(0, 3, 1, 2)
    t = t.to(DEVICE)
    t = (t - mean_t) / std_t
    kwargs = {'interpolate_pos_encoding': True} if SUPPORTS_INTERP else {}
    out = model(pixel_values=t, **kwargs)
    tokens = out.last_hidden_state.float().cpu().numpy()
    patch_tokens = tokens[:, 1 + NUM_REGISTER_TOKENS:]
    if patch_tokens.shape[1] != N_TOKENS:
        raise RuntimeError(f'Esperava {N_TOKENS} patch tokens, vieram {patch_tokens.shape[1]}.')
    return patch_tokens

## 6. Coleta de crops por site (positivos + negativos, com checkpoint)

**Positivos**: 1 crop centrado em cada polígono de daninha (qualquer tipo). Metadados incluem `geojson_source` original pra você identificar o tipo (folha_larga/estreita/mamona) na hora de revisar.

**Negativos**: amostra `MAX_CULTIVO_CROPS_PER_SITE` pontos aleatórios da plantação (respeitando `INVERTED_PLANTACAO` pra Flaviano), rejeitando os que caem em bbox de qualquer daninha (via `sindex`).

In [ ]:
def load_polys(path, raster_crs):
    if not os.path.exists(path):
        print(f'  aviso: arquivo nao encontrado, pulando: {path}')
        return []
    try:
        gdf = gpd.read_file(path)
    except Exception as e:
        print(f'  aviso: nao consegui ler {path}: {e}')
        return []
    if gdf.crs is not None and gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)
    polys = []
    for i, g in enumerate(gdf.geometry):
        if g is None or g.is_empty:
            continue
        if not g.is_valid:
            g = g.buffer(0)
        if g.is_empty:
            continue
        polys.append((i, g))
    return polys


def read_centered_crop(src, center_col, center_row):
    half = CROP_PIXELS // 2
    col_off = int(round(center_col - half))
    row_off = int(round(center_row - half))
    win = Window(col_off, row_off, CROP_PIXELS, CROP_PIXELS)
    tile = src.read([1, 2, 3], window=win, boundless=True, fill_value=0)
    return np.moveaxis(tile, 0, -1)   # (H, W, 3) uint8


def sample_cultivo_pixel_centers(src, plant_union, daninha_sindex, daninha_geoms,
                                 n_target, invert_plantacao, rng):
    """Sampleia n_target centros (col, row) dentro da plantacao SEM overlap com daninha."""
    if plant_union is None or plant_union.is_empty:
        return []
    minx, miny, maxx, maxy = plant_union.bounds
    if invert_plantacao:
        # area valida = FORA do poligono; usa bounds do raster inteiro
        minx, miny = src.transform * (0, src.height)
        maxx, maxy = src.transform * (src.width, 0)
    centers = []
    max_tries = n_target * 15   # margem folgada
    tries = 0
    while len(centers) < n_target and tries < max_tries:
        tries += 1
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        pt = Point(x, y)
        # dentro da plantacao?
        if invert_plantacao:
            if plant_union.contains(pt):
                continue
        else:
            if not plant_union.contains(pt):
                continue
        # rejeita se cair em daninha
        if daninha_sindex is not None:
            hits = list(daninha_sindex.query(pt, predicate='intersects'))
            if hits:
                continue
        col, row = ~src.transform * (x, y)
        # confirma que a janela cabe (com folga)
        half = CROP_PIXELS // 2
        if col < half or col > src.width - half or row < half or row > src.height - half:
            continue
        centers.append((float(col), float(row), float(x), float(y)))
    if len(centers) < n_target:
        print(f'  aviso: amostrei {len(centers)}/{n_target} cultivo apos {tries} tentativas')
    return centers


def collect_site_crops(cfg):
    print(f"\n=== {cfg['nome']} ===")
    invert_plantacao = cfg['nome'] in INVERTED_PLANTACAO
    if invert_plantacao:
        print(f'  aviso: {cfg["nome"]} em INVERTED_PLANTACAO -- area valida = FORA do poligono')
    rng = np.random.default_rng(RANDOM_STATE)
    crops_out = []

    with rasterio.open(cfg['imagem']) as src:
        raster_crs = src.crs
        raster_w, raster_h = src.width, src.height

        # 1. carregar daninhas (todos os geojsons colapsados)
        daninha_items = []   # (geojson_basename, poly_id, geom)
        for gj in cfg['geojsons']:
            polys = load_polys(gj, raster_crs)
            for poly_id, geom in polys:
                daninha_items.append((os.path.basename(gj), poly_id, geom))
            print(f'  daninha: {os.path.basename(gj)} -> {len(polys)} poligonos')
        print(f'  total daninha: {len(daninha_items)} poligonos')

        # 2. positivos: 1 crop por poligono
        pending_pos = []
        for gj_src, poly_id, geom in daninha_items:
            cx, cy = geom.centroid.x, geom.centroid.y
            col, row = ~src.transform * (cx, cy)
            if not (0 <= col < raster_w and 0 <= row < raster_h):
                continue
            minx, miny, maxx, maxy = geom.bounds
            pix_w = abs((maxx - minx) / src.transform.a)
            pix_h = abs((maxy - miny) / src.transform.e)
            if pix_w * pix_h < MIN_POLYGON_PIXELS:
                continue
            pending_pos.append((gj_src, poly_id, float(col), float(row), float(cx), float(cy)))
        print(f'  positivos apos filtros: {len(pending_pos)}')

        # 3. negativos: sampleia da plantacao
        plantacao_polys = load_polys(cfg['plantacao'], raster_crs) if cfg.get('plantacao') else []
        plant_union = unary_union([g.buffer(0) for _, g in plantacao_polys]) if plantacao_polys else None
        daninha_gs = gpd.GeoSeries([g for _, _, g in daninha_items]) if daninha_items else None
        daninha_sindex = daninha_gs.sindex if daninha_gs is not None else None
        neg_centers = sample_cultivo_pixel_centers(
            src, plant_union, daninha_sindex, daninha_gs,
            MAX_CULTIVO_CROPS_PER_SITE, invert_plantacao, rng)
        print(f'  negativos amostrados: {len(neg_centers)}')

        # 4. processa positivos + negativos em batches pela GPU
        BATCH_GPU = 32
        # positivos
        for i in tqdm(range(0, len(pending_pos), BATCH_GPU), desc=f'{cfg["nome"]} pos'):
            batch = pending_pos[i:i + BATCH_GPU]
            crops = np.stack([read_centered_crop(src, item[2], item[3]) for item in batch], axis=0)
            feats = extract_crop_features_batch(crops)
            for j, item in enumerate(batch):
                gj_src, poly_id, col, row, lon, lat = item
                crops_out.append({
                    'features': feats[j].astype(np.float16),
                    'label': 1,
                    'meta': {
                        'site': cfg['nome'],
                        'kind': 'positive',
                        'geojson_source': gj_src,
                        'polygon_id': poly_id,
                        'centroid_col': col, 'centroid_row': row,
                        'centroid_lon': lon, 'centroid_lat': lat,
                    },
                })
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
        # negativos
        for i in tqdm(range(0, len(neg_centers), BATCH_GPU), desc=f'{cfg["nome"]} neg'):
            batch = neg_centers[i:i + BATCH_GPU]
            crops = np.stack([read_centered_crop(src, item[0], item[1]) for item in batch], axis=0)
            feats = extract_crop_features_batch(crops)
            for j, (col, row, lon, lat) in enumerate(batch):
                crops_out.append({
                    'features': feats[j].astype(np.float16),
                    'label': 0,
                    'meta': {
                        'site': cfg['nome'],
                        'kind': 'negative',
                        'geojson_source': '',
                        'polygon_id': -1,
                        'centroid_col': col, 'centroid_row': row,
                        'centroid_lon': lon, 'centroid_lat': lat,
                    },
                })
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    n_pos = sum(1 for c in crops_out if c['label'] == 1)
    n_neg = sum(1 for c in crops_out if c['label'] == 0)
    print(f'  crops finais: daninha={n_pos} cultivo={n_neg}')
    return crops_out


def config_signature_site(cfg):
    geojson_stats = []
    all_files = list(cfg['geojsons']) + ([cfg['plantacao']] if cfg.get('plantacao') else [])
    for gj in all_files:
        if os.path.exists(gj):
            st = os.stat(gj)
            geojson_stats.append((os.path.basename(gj), st.st_size, int(st.st_mtime)))
        else:
            geojson_stats.append((os.path.basename(gj), 0, 0))
    payload = repr((
        'dinov3-daninhabin', DINO_MODEL, CROP_PIXELS, PATCH, DINOV3_MEAN, DINOV3_STD,
        MIN_POLYGON_PIXELS, MAX_CULTIVO_CROPS_PER_SITE, RANDOM_STATE,
        cfg['nome'], geojson_stats,
    ))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def collect_site_crops_cached(cfg):
    sig = config_signature_site(cfg)
    ck_path = f"{CHECKPOINT_DIR}/{cfg['nome']}_{sig}.joblib"
    if os.path.exists(ck_path):
        d = joblib.load(ck_path)
        n_pos = sum(1 for c in d if c['label'] == 1)
        n_neg = sum(1 for c in d if c['label'] == 0)
        print(f"  (cache: {cfg['nome']} -- daninha={n_pos} cultivo={n_neg})")
        return d
    crops = collect_site_crops(cfg)
    joblib.dump(crops, ck_path)
    return crops

## 7. Rodar coleta pra todos os sites

In [ ]:
all_crops = []
for cfg in PARES_CONFIG:
    all_crops.extend(collect_site_crops_cached(cfg))

n_pos = sum(1 for c in all_crops if c['label'] == 1)
n_neg = sum(1 for c in all_crops if c['label'] == 0)
print(f'\nTotal: {len(all_crops)} crops | daninha={n_pos} cultivo={n_neg}')
assert n_pos > 0 and n_neg > 0, 'Preciso de positivos E negativos pra treinar bin.'

## 8. Split treino/validação (faixa espacial dentro de cada fazenda)

Split respeita positivos E negativos: pra cada site, ordena todos os crops pelo centroide em pixels no eixo `SPLIT_AXIS`. 20% mais à direita = val, gap de `MARGIN_TILES * CROP_PIXELS` px.

In [ ]:
def split_train_val(all_crops, val_fraction, margin_tiles, axis='x'):
    if axis not in ('x', 'y'):
        raise ValueError(f"SPLIT_AXIS invalido: {axis!r}")
    key = 'centroid_col' if axis == 'x' else 'centroid_row'
    margin_px = margin_tiles * CROP_PIXELS

    train, val = [], []
    by_site = {}
    for c in all_crops:
        by_site.setdefault(c['meta']['site'], []).append(c)

    for site, crops in by_site.items():
        crops_sorted = sorted(crops, key=lambda c: c['meta'][key])
        total = len(crops_sorted)
        if total == 0:
            continue
        n_val_target = int(round(total * val_fraction))
        if n_val_target == 0:
            train.extend(crops_sorted)
            print(f'  {site}: total={total} val=0 (fraction {val_fraction} muito pequena)')
            continue
        val_start_idx = total - n_val_target
        val_start_pos = crops_sorted[val_start_idx]['meta'][key]
        train_end_pos = val_start_pos - margin_px

        n_tr = n_va = n_gap = 0
        n_tr_pos = n_va_pos = 0
        for c in crops_sorted:
            p = c['meta'][key]
            if p >= val_start_pos:
                val.append(c); n_va += 1
                if c['label'] == 1: n_va_pos += 1
            elif p < train_end_pos:
                train.append(c); n_tr += 1
                if c['label'] == 1: n_tr_pos += 1
            else:
                n_gap += 1
        print(f'  {site}: treino={n_tr} (pos={n_tr_pos}) | val={n_va} (pos={n_va_pos}) | gap={n_gap}')
    return train, val


train_crops, val_crops = split_train_val(all_crops, VAL_FRACTION, MARGIN_TILES, SPLIT_AXIS)
print(f'\nTotal treino: {len(train_crops)} | val: {len(val_crops)}')

# Pesos de classe
if CLASS_WEIGHTS is None:
    counts = np.zeros(N_CLASSES, dtype=np.float64)
    for c in train_crops:
        counts[c['label']] += 1
    counts_safe = np.maximum(counts, 1.0)
    weights = counts.sum() / (N_CLASSES * counts_safe)
    weights = np.clip(weights, 0.1, 20.0)
    class_weights = weights.astype(np.float32)
else:
    class_weights = np.array(CLASS_WEIGHTS, dtype=np.float32)
print('Pesos por classe: ' + ', '.join(f'{CLASS_NAMES[i]}={class_weights[i]:.2f}' for i in range(N_CLASSES)))

## 9. Dataset e DataLoader

In [ ]:
class CropFeaturesDataset(Dataset):
    def __init__(self, crops):
        self.crops = crops

    def __len__(self):
        return len(self.crops)

    def __getitem__(self, idx):
        c = self.crops[idx]
        feats = torch.from_numpy(c['features'].astype(np.float32))
        label = torch.tensor(c['label'], dtype=torch.long)
        return feats, label


train_loader = DataLoader(CropFeaturesDataset(train_crops), batch_size=BATCH, shuffle=True, drop_last=False)
val_loader = None
if len(val_crops) > 0:
    val_loader = DataLoader(CropFeaturesDataset(val_crops), batch_size=BATCH, shuffle=False, drop_last=False)
print(f'batches treino: {len(train_loader)}' + (f' | val: {len(val_loader)}' if val_loader else ''))

## 10. Head Transformer + mean pool → Linear(1024, 2)

Mesma receita do notebook base (LayerNorm + pos_encoding + 2× TransformerEncoderLayer GELU norm_first), com mean pool no fim pra colapsar os 196 tokens.

In [ ]:
class CropTransformerHead(nn.Module):
    def __init__(self, hidden_size, n_tokens, n_layers=2, nhead=8, dropout=0.1,
                 classifier_dropout=0.0, n_classes=2):
        super().__init__()
        self.input_norm = nn.LayerNorm(hidden_size)
        self.pos_encoding = nn.Parameter(torch.zeros(1, n_tokens, hidden_size))
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=nhead, dim_feedforward=hidden_size * 2,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.class_dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, tokens):
        x = self.input_norm(tokens) + self.pos_encoding
        x = self.encoder(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.class_dropout(x)
        return self.classifier(x)


head = CropTransformerHead(
    hidden_size=HIDDEN_SIZE, n_tokens=N_TOKENS, n_layers=TRANSFORMER_LAYERS,
    nhead=NHEAD, dropout=TRANSFORMER_DROPOUT, classifier_dropout=CLASSIFIER_DROPOUT,
    n_classes=N_CLASSES,
).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f'Head transformer: {n_params:,} parametros treinaveis')

## 11. Loss, otimizador, scheduler

In [ ]:
class_weight_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weight_t, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch_idx):
    if WARMUP_EPOCHS > 0 and epoch_idx < WARMUP_EPOCHS:
        return (epoch_idx + 1) / WARMUP_EPOCHS
    denom = max(EPOCHS - WARMUP_EPOCHS, 1)
    progress = (epoch_idx - WARMUP_EPOCHS) / denom
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def run_epoch(loader, train=True):
    head.train(mode=train)
    total_loss, n_batches = 0.0, 0
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = head(tokens)
            loss = loss_fn(logits, labels)
        if train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(loader):
    """Retorna (f1_daninha, y_true, y_pred). f1_daninha usado pro early stop."""
    if loader is None:
        return None, None, None
    head.eval()
    all_true, all_pred = [], []
    for tokens, labels in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)
        pred = logits.argmax(dim=-1).cpu().numpy()
        all_true.append(labels.numpy())
        all_pred.append(pred)
    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    f1_daninha = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    return f1_daninha, y_true, y_pred

## 12. Loop de treino (early stop em F1 da classe daninha)

In [ ]:
best_f1 = -1.0
best_state = None
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_f1, _, _ = evaluate(val_loader)
    current_lr = optimizer.param_groups[0]['lr']
    msg = f'epoca {epoch}/{EPOCHS} | LR: {current_lr:.2e} | loss treino: {train_loss:.4f}'
    if val_f1 is not None:
        msg += f' | F1 daninha (val): {val_f1:.4f}'
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            epochs_no_improve = 0
            msg += '  (melhor, salvando)'
        else:
            epochs_no_improve += 1
    else:
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    print(msg)
    scheduler.step()
    if val_loader is not None and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'early stop -- {EARLY_STOP_PATIENCE} epocas sem melhora.')
        break

if best_state is not None:
    head.load_state_dict(best_state)
print('Treino concluido.' + (f' Melhor F1 daninha (val): {best_f1:.4f}' if best_f1 >= 0 else ''))

torch.save({
    'head_state_dict': head.state_dict(),
    'hidden_size': HIDDEN_SIZE, 'n_tokens': N_TOKENS, 'n_classes': N_CLASSES,
    'transformer_layers': TRANSFORMER_LAYERS, 'nhead': NHEAD,
    'transformer_dropout': TRANSFORMER_DROPOUT, 'classifier_dropout': CLASSIFIER_DROPOUT,
    'label_smoothing': LABEL_SMOOTHING, 'class_weights': class_weights.tolist(),
    'dino_model': DINO_MODEL, 'dinov3_mean': DINOV3_MEAN, 'dinov3_std': DINOV3_STD,
    'crop_pixels': CROP_PIXELS, 'patch': PATCH, 'num_register_tokens': NUM_REGISTER_TOKENS,
    'class_names': CLASS_NAMES, 'best_val_f1_daninha': best_f1, 'ciclo': CICLO,
}, OUTPUT_MODEL)
print(f'Modelo salvo em {OUTPUT_MODEL}')

## 13. Relatório de métricas

In [ ]:
report_lines = [f'=== ciclo {CICLO} | DINOv3+Transformer per-crop binario (daninha vs cultivo) ===']
report_lines.append(f'Split: {VAL_FRACTION:.0%} da faixa "{SPLIT_AXIS}" por site, margem={MARGIN_TILES} tiles ({MARGIN_TILES*CROP_PIXELS}px)')
n_pos_all = sum(1 for c in all_crops if c['label'] == 1)
n_neg_all = sum(1 for c in all_crops if c['label'] == 0)
report_lines.append(f'Total crops: {len(all_crops)} (daninha={n_pos_all}, cultivo={n_neg_all})')
report_lines.append(f'Treino: {len(train_crops)} | Val: {len(val_crops)}')
report_lines.append('')

if val_loader is not None:
    val_f1, y_true, y_pred = evaluate(val_loader)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    accuracy = (y_true == y_pred).mean()
    report_lines.append(f'F1 daninha (val): {val_f1:.4f}')
    report_lines.append(f'Precision daninha: {prec:.4f}')
    report_lines.append(f'Recall daninha: {rec:.4f}')
    report_lines.append(f'Accuracy geral: {accuracy:.4f}')
    report_lines.append('')
    report_lines.append(classification_report(y_true, y_pred, target_names=['cultivo', 'daninha'], zero_division=0))
    report_lines.append('Matriz de confusao [linhas=verdadeiro, colunas=previsto], ordem [cultivo, daninha]:')
    report_lines.append(str(confusion_matrix(y_true, y_pred, labels=[0, 1])))
else:
    report_lines.append('sem tiles de validacao -- relatorio de metricas pulado.')

report_text = '\n'.join(report_lines)
print(report_text)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(f'\nRelatorio salvo em {REPORT_PATH}')

## 14. Exportar suspeitas → CSV pra revisão humana

**Só sobre os POSITIVOS anotados** (crops de daninha do dataset). Negativos são amostrados aleatoriamente, então erro do modelo neles não indica anotação errada.

- **`misclass`**: positivo anotado como daninha mas predito como cultivo → provavelmente o polígono está errado (não é daninha real, ou é sliver, ou área de cultivo saudável marcada por engano). Ordenado por `prob_daninha` crescente (mais confiante de que é cultivo).
- **`low_conf`**: positivo predito daninha mas com `prob_daninha < SUSPECT_LOW_CONF_THRESHOLD` → o modelo hesita. Geometria ambígua.

Cada linha aponta pro polígono via `site + geojson_source + polygon_id`. `centroid_lon/lat` pra abrir no QGIS.

In [ ]:
@torch.no_grad()
def infer_all(crops):
    head.eval()
    loader = DataLoader(CropFeaturesDataset(crops), batch_size=BATCH, shuffle=False)
    preds, probs_all = [], []
    for tokens, _ in loader:
        tokens = tokens.to(DEVICE, non_blocking=True)
        logits = head(tokens)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds.append(probs.argmax(axis=-1))
        probs_all.append(probs)
    return np.concatenate(preds), np.concatenate(probs_all, axis=0)


# So os positivos anotados
positive_crops = [c for c in all_crops if c['meta']['kind'] == 'positive']
print(f'Analisando {len(positive_crops)} positivos anotados...')

pos_preds, pos_probs = infer_all(positive_crops)
prob_daninha = pos_probs[:, 1]

suspects = []
for i, c in enumerate(positive_crops):
    pred = int(pos_preds[i])
    pd = float(prob_daninha[i])
    if pred != 1:
        kind = 'misclass'
    elif pd < SUSPECT_LOW_CONF_THRESHOLD:
        kind = 'low_conf'
    else:
        continue
    suspects.append({
        **c['meta'],
        'true_class': 'daninha',
        'pred_class': CLASS_NAMES[pred],
        'prob_daninha': pd,
        'prob_cultivo': float(pos_probs[i, 0]),
        'kind': kind,
    })

# misclass primeiro (por prob_daninha crescente = mais confiante que e' cultivo),
# depois low_conf (por prob_daninha crescente = mais duvidoso)
suspects.sort(key=lambda s: (0 if s['kind'] == 'misclass' else 1, s['prob_daninha']))

n_mis = sum(1 for s in suspects if s['kind'] == 'misclass')
n_lc = sum(1 for s in suspects if s['kind'] == 'low_conf')
print(f'Suspeitas: {len(suspects)} ({n_mis} misclass, {n_lc} low_conf) de {len(positive_crops)} positivos')

if suspects:
    fieldnames = ['kind', 'site', 'geojson_source', 'polygon_id',
                  'true_class', 'pred_class', 'prob_daninha', 'prob_cultivo',
                  'centroid_lon', 'centroid_lat', 'centroid_col', 'centroid_row']
    with open(SUSPECTS_CSV, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for s in suspects:
            w.writerow({k: s.get(k, '') for k in fieldnames})
    print(f'Suspeitas salvas em {SUSPECTS_CSV}')
    print('\nTop 10 suspeitas:')
    for s in suspects[:10]:
        print(f"  [{s['kind']}] {s['site']}/{s['geojson_source']}#{s['polygon_id']} "
              f"pred={s['pred_class']} prob_daninha={s['prob_daninha']:.3f}")

## 15. Próximo ciclo

1. Abrir `suspeitas_c1.csv` (ou QGIS pelas coords `centroid_lon/lat`) e revisar as manchas marcadas.
2. Corrigir no Drive: deletar polígono errado, ajustar geometria, mover pra outro geojson.
3. Bump `CICLO = 2` no topo, rodar tudo de novo.
4. **Cache é inteligente**: assinatura por site inclui `mtime` dos geojsons — só o site cujo geojson mudou recalcula features. Outros voltam do cache em segundos.
5. Comparar `relatorio_c2.txt` vs `_c1`: F1 daninha subiu? Recall subiu sem perder precision? Número de suspeitas caiu? → dataset melhorou.

**Regra**: se o modelo é bem calibrado (mesmo instrumento, mesmos params), qualquer melhora vem de rótulos melhores — não de tuning.